<a href="https://colab.research.google.com/github/teemaphorn/delivery-project/blob/main/Project_Parcel_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
#ส่วนรับเข้าพัสดุทีละชิ้น
import random
import pandas as pd

# 1. ประกาศ Class สำหรับเก็บข้อมูลองค์ประกอบต่างๆ

class Customer:
    # เก็บข้อมูลลูกค้า (ผู้ส่ง/ผู้รับ)
    def __init__(self, customer_id, name, phone):
        self.customer_id = customer_id
        self.name = name
        self.phone = phone

class Parcel:
    # เก็บข้อมูลพัสดุและกำหนดประเภทบริการเริ่มต้น
    def __init__(self, parcel_id, weight_kg, service_type="ธรรมดา"):
        self.parcel_id = parcel_id
        self.weight_kg = weight_kg
        self.service_type = service_type

class ParcelIntake:
    # รวมข้อมูลการรับพัสดุ 1 รายการ
    def __init__(self, intake_id, sender_obj, recipient_obj, parcel_obj, distance_km):
        self.intake_id = intake_id
        self.sender = sender_obj
        self.recipient = recipient_obj
        self.parcel = parcel_obj
        self.distance_km = distance_km
        self.status = "รับแล้ว"

    def get_summary(self):
        # แปลงเป็น dict เพื่อเตรียมเอาไปสร้าง DataFrame
        return {
            "intake_id": self.intake_id,
            "parcel_id": self.parcel.parcel_id,
            "sender_name": self.sender.name,
            "recipient_name": self.recipient.name,
            "weight_kg": self.parcel.weight_kg,
            "service_type": self.parcel.service_type,
            "distance_km": self.distance_km,
            "status": self.status
        }


# 2. ฟังก์ชันช่วยสุ่มข้อมูลตัวอย่าง

def generate_customer_name():
    # สุ่มชื่อ-นามสกุล
    first_names = ["เรยา", "มุนินทร์", "มุตตา", "เรณู", "พิไล", "ทองดี"]
    last_names = ["วงศ์เสวต", "จงสวัสดิ์", "จงสวัสดิ์", "ตาคลี", "อัศวรุ่งเรืองกิจ", "บุญพูลทรัพย์ "]
    return f"{random.choice(first_names)} {random.choice(last_names)}"

def generate_random_weight(min_w=0.5, max_w=15.0):
    # สุ่มน้ำหนักพัสดุ (ทศนิยม 2 ตำแหน่ง)
    return round(random.uniform(min_w, max_w), 2)

def generate_random_distance(min_d=1.0, max_d=250.0):
    # สุ่มระยะทาง (ทศนิยม 1 ตำแหน่ง)
    return round(random.uniform(min_d, max_d), 1)


# 3. ส่วนสร้างข้อมูล 300 รายการและบันทึกไฟล์

random.seed(42)  # ล็อกค่า random ให้ผลลัพธ์คงที่
intake_records = []
service_options = ["ธรรมดา", "ด่วนพิเศษ (Express)", "แช่เย็น (Cold Chain)"]

# วนลูปสร้างออเดอร์ 300 รายการ
for i in range(1, 301):
    sender = Customer(f"CUS-S{i:03d}", generate_customer_name(), f"08{random.randint(10000000, 99999999)}")
    recipient = Customer(f"CUS-R{i:03d}", generate_customer_name(), f"09{random.randint(10000000, 99999999)}")

    parcel_item = Parcel(
        parcel_id=f"TH{i:05d}",
        weight_kg=generate_random_weight(),
        service_type=random.choice(service_options)
    )

    intake_item = ParcelIntake(
        intake_id=f"INT-{i:04d}",
        sender_obj=sender,
        recipient_obj=recipient,
        parcel_obj=parcel_item,
        distance_km=generate_random_distance()
    )

    intake_records.append(intake_item.get_summary())

# แปลงเป็น DataFrame และเซฟลง CSV
df_intake = pd.DataFrame(intake_records)
df_intake.to_csv("parcel_intake_300.csv", index=False, encoding="utf-8-sig")

print(f"การรับพัสดุเข้าระบบสำเร็จทั้งหมด: {len(df_intake)} รายการ")
df_intake.head()

การรับพัสดุเข้าระบบสำเร็จทั้งหมด: 300 รายการ


,intake_id,parcel_id,sender_name,recipient_name,weight_kg,service_type,distance_km,status
0,INT-0001,TH00001,ทองดี วงศ์เสวต,ทองดี จงสวัสดิ์,3.74,แช่เย็น (Cold Chain),26.5,รับแล้ว
1,INT-0002,TH00002,ทองดี อัศวรุ่งเรืองกิจ,พิไล ตาคลี,0.93,ธรรมดา,58.9,รับแล้ว
2,INT-0003,TH00003,พิไล วงศ์เสวต,มุนินทร์ บุญพูลทรัพย์,10.67,ด่วนพิเศษ (Express),55.9,รับแล้ว
3,INT-0004,TH00004,พิไล จงสวัสดิ์,มุนินทร์ บุญพูลทรัพย์,5.43,ธรรมดา,54.6,รับแล้ว
4,INT-0005,TH00005,มุตตา วงศ์เสวต,เรณู วงศ์เสวต,12.79,แช่เย็น (Cold Chain),66.9,รับแล้ว


In [14]:
import random
import pandas as pd

def calculate_shipping_fee(weight_kg, distance_km, service_type="ธรรมดา"):
    """
    ฟังก์ชันคำนวณค่าจัดส่งพัสดุ
    - weight_kg: น้ำหนักพัสดุ (กิโลเมตร)
    - distance_km: ระยะทางจัดส่ง (กิโลเมตร)
    - service_type: ประเภทบริการ (มี Default Value เป็น 'ธรรมดา')
    """
    base_fee = 15  # ค่าเปิดบิล(บาท)

# 1. คำนวณค่าบริการตามน้ำหนัก
    if weight_kg <= 1.0:
        weight_fee = 10                         # ไม่เกิน 1 kg คิด 10 บาท
    elif weight_kg <= 5.0:
        weight_fee = 10 + ((weight_kg - 1.0) * 5) # เกิน 1kg คิดเพิ่มkgละ5บาท
    else:
        weight_fee = 30 + ((weight_kg - 5.0) * 8) # เกิน 5 kg คิดเพิ่มkgละ8บาท

# 2. คำนวณค่าระยะทางแบบแบ่งโซน (Zone-based)

    if distance_km <= 15:
        distance_fee = 5
    elif distance_km <= 50:
        distance_fee = 15
    elif distance_km <= 150:
        distance_fee = 30
    else:
        distance_fee = 50



การคำนวณค่าจัดส่งตามน้ำหนัก/ระยะทางทั้งหมด 300 รายการเสร็จสมบูรณ์แล้ว


,intake_id,parcel_id,sender_name,recipient_name,weight_kg,distance_km,service_type,shipping_fee
0,INT-0001,TH00001,ทองดี วงศ์เสวต,ทองดี จงสวัสดิ์,3.74,26.5,แช่เย็น (Cold Chain),53.70
1,INT-0002,TH00002,ทองดี อัศวรุ่งเรืองกิจ,พิไล ตาคลี,0.93,58.9,ธรรมดา,55.00
2,INT-0003,TH00003,พิไล วงศ์เสวต,มุนินทร์ บุญพูลทรัพย์,10.67,55.9,ด่วนพิเศษ (Express),120.36
3,INT-0004,TH00004,พิไล จงสวัสดิ์,มุนินทร์ บุญพูลทรัพย์,5.43,54.6,ธรรมดา,78.44
4,INT-0005,TH00005,มุตตา วงศ์เสวต,เรณู วงศ์เสวต,12.79,66.9,แช่เย็น (Cold Chain),137.32
5,INT-0006,TH00006,เรยา บุญพูลทรัพย์,พิไล วงศ์เสวต,1.64,207.5,ด่วนพิเศษ (Express),78.20
6,INT-0007,TH00007,พิไล จงสวัสดิ์,มุนินทร์ บุญพูลทรัพย์,1.16,193.5,ธรรมดา,75.80
7,INT-0008,TH00008,เรยา จงสวัสดิ์,เรณู จงสวัสดิ์,9.72,41.5,ด่วนพิเศษ (Express),97.76
8,INT-0009,TH00009,มุตตา จงสวัสดิ์,มุตตา บุญพูลทรัพย์,1.54,43.6,แช่เย็น (Cold Chain),42.70
9,INT-0010,TH00010,ทองดี จงสวัสดิ์,เรณู ตาคลี,14.85,172.4,แช่เย็น (Cold Chain),173.80


In [12]:
import random
import pandas as pd


# 1. ประกาศ Class ParcelStatusTracker ก่อนใช้งาน
class ParcelStatusTracker:

    def __init__(self, intake_id, parcel_id, recipient_name, old_status):
        self.intake_id = intake_id
        self.parcel_id = parcel_id
        self.recipient_name = recipient_name
        self.old_status = old_status
        self.new_status = old_status

    def update_status(self, target_status):
        valid_statuses = ["รับแล้ว", "กำลังส่ง", "ส่งสำเร็จ"]
        if target_status in valid_statuses:
            self.new_status = target_status


# ---  วนลูปสุ่มอัปเดตสถานะใหม่ ---
status_options = ["รับแล้ว", "กำลังส่ง", "ส่งสำเร็จ"]
updated_statuses = []

for idx, row in df_intake.iterrows():
    tracker = ParcelStatusTracker(
        intake_id=row["intake_id"],
        parcel_id=row["parcel_id"],
        recipient_name=row["recipient_name"],
        old_status=row["status"],
    )

    next_status = random.choice(status_options)
    tracker.update_status(next_status)
    updated_statuses.append(tracker.new_status)

# สร้าง df_result และเพิ่มคอลัมน์สถานะ
df_result = df_intake.copy()
df_result["old_status"] = df_result["status"]
df_result["updated_status"] = updated_statuses

# กำหนดการจัดเรียงคอลัมน์
columns_order = [
    "intake_id",
    "parcel_id",
    "sender_name",
    "recipient_name",
    "weight_kg",
    "service_type",
    "distance_km",
    "old_status",
    "updated_status",
]

df_result = df_result[columns_order]

# บันทึกไฟล์ CSV
df_result.to_csv("parcel_tracked_300.csv", index=False, encoding="utf-8-sig")

print(f"การรับพัสดุและอัปเดตสถานะสำเร็จทั้งหมด: {len(df_result)} รายการ")
df_result

การรับพัสดุและอัปเดตสถานะสำเร็จทั้งหมด: 300 รายการ


,intake_id,parcel_id,sender_name,recipient_name,weight_kg,service_type,distance_km,old_status,updated_status
0,INT-0001,TH00001,ทองดี วงศ์เสวต,ทองดี จงสวัสดิ์,3.74,แช่เย็น (Cold Chain),26.5,รับแล้ว,ส่งสำเร็จ
1,INT-0002,TH00002,ทองดี อัศวรุ่งเรืองกิจ,พิไล ตาคลี,0.93,ธรรมดา,58.9,รับแล้ว,ส่งสำเร็จ
2,INT-0003,TH00003,พิไล วงศ์เสวต,มุนินทร์ บุญพูลทรัพย์,10.67,ด่วนพิเศษ (Express),55.9,รับแล้ว,กำลังส่ง
3,INT-0004,TH00004,พิไล จงสวัสดิ์,มุนินทร์ บุญพูลทรัพย์,5.43,ธรรมดา,54.6,รับแล้ว,กำลังส่ง
4,INT-0005,TH00005,มุตตา วงศ์เสวต,เรณู วงศ์เสวต,12.79,แช่เย็น (Cold Chain),66.9,รับแล้ว,กำลังส่ง
...,...,...,...,...,...,...,...,...,...
295,INT-0296,TH00296,เรณู อัศวรุ่งเรืองกิจ,มุนินทร์ ตาคลี,4.20,ด่วนพิเศษ (Express),162.0,รับแล้ว,กำลังส่ง
296,INT-0297,TH00297,เรณู จงสวัสดิ์,พิไล บุญพูลทรัพย์,1.25,ด่วนพิเศษ (Express),168.7,รับแล้ว,ส่งสำเร็จ
297,INT-0298,TH00298,พิไล วงศ์เสวต,มุนินทร์ อัศวรุ่งเรืองกิจ,3.65,ธรรมดา,217.5,รับแล้ว,ส่งสำเร็จ
298,INT-0299,TH00299,มุตตา จงสวัสดิ์,มุนินทร์ จงสวัสดิ์,11.33,ด่วนพิเศษ (Express),228.7,รับแล้ว,รับแล้ว
